# Семантический поиск: Sentence Transformers + FAISS

Цель — найти похожие клинические случаи по `description`. Единица поиска — одна строка корпуса `data/processed/prepared_dataset.csv`; принадлежность строк одному пациенту пока не проверена. `bethesda` и `conclusion` используются только для просмотра найденных случаев, а не для построения векторов. Классификацию Bethesda здесь не решаем.

Пользователь запускает ноутбук локально в проектной `.venv`. Агент не читает медицинские данные и запросы и не запускает ноутбук. Удаление программного флага не подтверждает техническую изоляцию; ограничения запуска агентом из AGENTS.md сохраняются. Корпус остаётся неизменным. Индекс FAISS остаётся на CPU; ниже предусмотрены его экспорт с JSON метаданных и запись индекса и CSV выдачи только в локальный MLflow http://127.0.0.1:5000. Не сохраняйте ноутбук с выводом реальных данных.

## 1. Условия запуска и зависимости

Выберите ядро локальной `.venv`. `sentence-transformers` и `faiss-cpu` уже перечислены в `requirements.txt`. Этот ноутбук ничего не устанавливает. Программный флаг подтверждения изоляции удалён: он не обеспечивал техническую защиту данных.

При пользовательском запуске `ALLOW_MODEL_DOWNLOAD=True` разрешает скачать недостающие файлы закреплённой версии модели с Hugging Face в локальный кеш. Если файлы уже доступны, они используются из кеша; разрешение скачивания допускает обращения к Hugging Face. Для работы без сети установите `ALLOW_MODEL_DOWNLOAD=False`: `local_files_only=True` разрешит использовать только кеш. При неполном кеше загрузка завершится ошибкой. Кодирование описаний и запросов выполняется локально на одном выбранном CPU или CUDA; тексты не передаются в облачный inference API.

In [ ]:
import re
import random
import hashlib
import json
import sys

from pathlib import Path
from importlib.metadata import version

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'requirements.txt').is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError('Не найден корень проекта с requirements.txt.')
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.common.dataset_versioning import calculate_sha256

print({name: version(name) for name in ('sentence-transformers', 'faiss-cpu', 'numpy', 'pandas')})


## 2. Загрузка корпуса

Корпус — неизменяемый `data/processed/prepared_dataset.csv`. После определения `PROJECT_ROOT` добавляем корень в `sys.path`, чтобы импортировать `calculate_sha256` из `src/common/dataset_versioning.py`. Функция принимает `Path`, читает файл блоками и возвращает полный SHA256 — 64 шестнадцатеричных символа. Перед `pd.read_csv` сохраняем его в `DATASET_VERSION`; копии и версии файла не создаём. Хеш относится к байтам исходного CSV, а не только к выбранным строкам.

`pd.read_csv` возвращает DataFrame. `usecols` ограничивает чтение четырьмя полями: case_id, description, conclusion и bethesda. case_id связывает результат со строкой корпуса; source_table, source_row, duplicate_group и duplicate_type не загружаются. `sep=';'`, `encoding='utf-8-sig'` и `dtype={'case_id': 'string'}` совпадают с `test_TF-IDF.ipynb`: case_id сохраняется строкой, типы остальных полей определяет pandas. Схема этого CSV агентом не проверялась.

Проверяем пустой корпус, отсутствующие или состоящие из пробелов case_id и повторяющиеся case_id во всём загруженном корпусе, до отбора описаний. При ошибке останавливаемся без очистки и дедупликации. Выводим только полный SHA256 и число строк, без таблицы и значений идентификаторов. Агент не читает корпус и запросы и не выполняет эти ячейки.

case_id — связь со строкой корпуса; внутри свободного текста также могут встречаться идентификаторы. Перед показом выдачи это нужно проверить локально в рамках разрешённой процедуры.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'prepared_dataset.csv'
RESULT_COLUMNS = ['case_id', 'description', 'conclusion', 'bethesda']
DATASET_VERSION = calculate_sha256(Path(DATA_PATH))

cases = pd.read_csv(
    DATA_PATH, sep=';', encoding='utf-8-sig',
    usecols=RESULT_COLUMNS, dtype={'case_id': 'string'},
)
if cases.empty:
    raise ValueError('Поисковый корпус пуст.')
if cases['case_id'].isna().any() or cases['case_id'].str.strip().eq('').any():
    raise ValueError('В корпусе есть пустые case_id; загрузка остановлена.')
if cases['case_id'].duplicated().any():
    raise ValueError('В корпусе есть повторяющиеся case_id; загрузка остановлена.')

print({'dataset_version_sha256': DATASET_VERSION, 'corpus_cases': len(cases)})


## 3. Подготовка описаний

Как в `test_TF-IDF.ipynb`, оставляем строки, где description не пропущен и не состоит только из пробелов. `str.strip()` используется только в условии отбора: в `search_cases` сохраняется исходный description, включая пробелы по краям. Дополнительной очистки нет. При отсутствии подходящих описаний останавливаемся. Выводим только число индексируемых случаев и число исключённых пустых описаний.

Булева маска и `reset_index(drop=True)` сохраняют относительный порядок строк CSV и задают позиции от нуля. Вектор с номером `i` соответствует `search_cases.iloc[i]` и её case_id. После построения индекса порядок таблицы менять нельзя. При одинаковом файле и успешных проверках оба метода получают одинаковые строки и исходные описания.

Дубликаты не удаляем автоматически: похожие описания могут принадлежать разным случаям. Их влияние на оценку и повторные случаи одного пациента нужно исследовать отдельно.

In [ ]:
has_description = cases['description'].notna()
has_description &= cases['description'].str.strip().ne('')

search_cases = cases.loc[has_description].copy().reset_index(drop=True)

if search_cases.empty:
    raise ValueError('В корпусе нет непустых описаний для поиска.')
print({'index_cases': len(search_cases), 'excluded_empty': len(cases) - len(search_cases)})


## 4. Модель представления текста

Embedding — плотный вектор, представляющий текст. Начинаем с `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`: готовой многоязычной модели для сходства текстов. Она возвращает 384-мерные векторы и не требует обучения на нашем корпусе. Это стартовый вариант, а не подтверждённо лучшая модель для цитологии. Альтернатива — более тяжёлая многоязычная retrieval-модель; сравнивать её нужно на тех же запросах и ручной разметке.

`SentenceTransformer` загружает encoder; DEVICE выбирает CUDA при доступности через PyTorch, иначе CPU. Это устройство encoder; FAISS остаётся на CPU. `MODEL_REVISION` закреплён строкой `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`, полученной из [публичных метаданных main Hugging Face](https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/revision/main). Ноутбук не определяет main заново при каждом запуске. Полный SHA содержит ровно 40 шестнадцатеричных символов; проверяем его до загрузки. `trust_remote_code=False` запрещает выполнение пользовательского кода из репозитория модели. [Карточка модели](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2).

`local_files_only=not ALLOW_MODEL_DOWNLOAD` связывает загрузку с выбранным режимом. При ошибке выводим пояснение и исходную причину; модель, revision и настройки автоматически не меняем. После успешной загрузки показываем название, SHA, фактическое устройство, размерность эмбеддинга и max_seq_length; при CUDA — также название GPU. Ошибка выбранной GPU не вызывает автоматический переход на CPU. Фактическая загрузка при подготовке этой версии ноутбука не проверялась.

Модель обрезает длинные тексты до `max_seq_length` токенов (частей слов), а не символов. Для этой модели штатный предел — 128; увеличение числа без проверки качества не решает проблему автоматически.


CPU — штатный режим, если CUDA недоступна. У CPU-only сборки PyTorch CUDA нет: одного DEVICE недостаточно для GPU; подготовка другой сборки требует отдельного согласования. Seed=42 задаём для random, NumPy и PyTorch до загрузки модели и кодирования. Это не гарантирует побитового совпадения CPU и GPU; устройство и seed фиксируются в run.


In [ ]:
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODEL_REVISION = 'e8f8c211226b894fcb81acc59f3b34ba3efd5f42'
ALLOW_MODEL_DOWNLOAD = True
BATCH_SIZE = 32
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)
elif torch.version.cuda is None:
    print('PyTorch установлен без CUDA: используется штатный режим CPU; пакеты не изменяются.')

if not isinstance(MODEL_REVISION, str) or not re.fullmatch(r'[0-9a-fA-F]{40}', MODEL_REVISION):
    raise ValueError('MODEL_REVISION должен быть полным 40-символьным commit SHA.')

model = None
try:
    model = SentenceTransformer(
        MODEL_NAME, revision=MODEL_REVISION, device=DEVICE,
        local_files_only=not ALLOW_MODEL_DOWNLOAD, trust_remote_code=False,
    )
except Exception as exc:
    mode = 'скачивание разрешено' if ALLOW_MODEL_DOWNLOAD else 'только локальный кеш'
    raise RuntimeError(
        f'Не удалось загрузить {MODEL_NAME}, revision={MODEL_REVISION}, device={DEVICE}, режим: {mode}. '
        'При разрешённом скачивании проверьте доступ к Hugging Face и место для кеша; '
        'в режиме без сети нужны все файлы этой revision в локальном кеше. '
        'Модель, revision и настройки не изменены. '
        f'Исходная ошибка: {type(exc).__name__}: {exc}'
    ) from exc

ACTUAL_DEVICE = str(model.device)
if model.device.type != torch.device(DEVICE).type:
    raise RuntimeError('Фактическое устройство отличается от выбранного; выполнение остановлено.')
GPU_NAME = torch.cuda.get_device_name(model.device) if model.device.type == 'cuda' else None
MODEL_SETTINGS = {
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'device': ACTUAL_DEVICE,
    'batch_size': BATCH_SIZE,
    'max_seq_length': model.max_seq_length,
    'embedding_dimension': model.get_embedding_dimension(),
    'random_seed': SEED,
}
if GPU_NAME is not None:
    MODEL_SETTINGS['gpu_name'] = GPU_NAME

model_info = {
    'model': MODEL_NAME,
    'revision': MODEL_REVISION,
    'device': str(model.device),
    'embedding_dimension': model.get_embedding_dimension(),
    'max_seq_length': model.max_seq_length,
}
if GPU_NAME is not None:
    model_info['gpu_name'] = GPU_NAME
print(model_info)


## 5. Векторы корпуса

`model.encode` принимает список описаний и возвращает NumPy-матрицу `(число случаев, размерность)`. `batch_size=32` задаёт число текстов за один проход и влияет на расход памяти. `normalize_embeddings=True` приводит каждый вектор к длине 1; `convert_to_numpy=True` подготавливает результат для FAISS. На этой симметричной модели корпус и запрос кодируем одинаковым методом, без специальных префиксов.

`np.ascontiguousarray(..., dtype=np.float32)` обеспечивает формат памяти, ожидаемый FAISS. Проверяем размеры, конечность чисел и нормировку, не показывая тексты или векторы.

In [ ]:
embeddings = model.encode(
    search_cases['description'].tolist(),
    batch_size=BATCH_SIZE, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(search_cases):
    raise ValueError('Размер матрицы не совпадает с поисковым корпусом.')
if not np.isfinite(embeddings).all():
    raise ValueError('Векторы содержат неконечные значения.')
if not np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5):
    raise ValueError('Векторы должны иметь единичную норму.')
print({'embedding_shape': embeddings.shape})


## 6. Хранение в FAISS

`faiss.IndexFlatIP(d)` создаёт точный индекс для векторов размерности `d`; `add` копирует в него векторы в исходном порядке. Индекс хранит сами векторы, но не описания: тексты берём по возвращённым позициям из `search_cases`.

IP означает скалярное произведение. Для векторов единичной длины оно равно косинусному сходству: больше — похожее. Значение не является вероятностью клинической релевантности. Точный поиск исключает ошибки приближённого индекса при первом эксперименте; альтернатива HNSW полезна позже при ограничениях скорости. [Описание индексов FAISS](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes).

Индекс строится один раз на CPU. При построении фиксируем настройки и упорядоченные case_id в INDEX_METADATA, а также отпечаток сериализованного индекса в памяти. Ниже faiss.write_index сохраняет именно этот индекс без повторного кодирования. Все хеши файлов вычисляются calculate_sha256; hashlib используется только для отпечатка индекса в памяти. JSON не содержит описаний или заключений. Загрузка сохранённого индекса для повторного поиска отдельным сценарием здесь не добавляется. [Сериализация FAISS](https://github.com/facebookresearch/faiss/wiki/Index-IO,-cloning-and-hyper-parameter-tuning).

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
if index.ntotal != len(search_cases):
    raise ValueError('Размер FAISS-индекса не совпадает с корпусом.')
INDEX_METADATA = {
    **MODEL_SETTINGS,
    'dataset_version_sha256': DATASET_VERSION,
    'embedding_dimension': index.d,
    'index_size_cases': index.ntotal,
    'normalize_embeddings': True,
    'faiss_index_type': type(index).__name__,
    'faiss_device': 'cpu',
    'case_ids': search_cases['case_id'].tolist(),
    'index_sha256': hashlib.sha256(faiss.serialize_index(index)).hexdigest(),
}
print({'faiss_vectors': index.ntotal, 'dimension': index.d})
del embeddings  # Векторы уже скопированы в FAISS; освобождаем вторую копию.


## 7. Поиск похожих случаев

`find_similar_cases(text, top_k=5)` принимает непустое описание и положительное целое `top_k`, возвращает DataFrame: `rank`, `similarity`, `case_id`, `description`, `conclusion`, `bethesda`. Bethesda и conclusion — справочные поля, не признаки поиска и не разметка релевантности. Запрос кодируется той же моделью и нормируется так же, как корпус.

`index.search(query_vector, k)` возвращает две матрицы: оценки и позиции соседей. Ограничиваем `k` числом случаев, чтобы FAISS не возвращал отсутствующие позиции `-1`. Функция ничего не печатает и не записывает. Результаты отсортированы по убыванию сходства; порядок при равных оценках не следует интерпретировать.

Если запрос взят из самого корпуса, собственная строка может оказаться первой. Это допустимо для диагностики, но не для независимой оценки качества. Для сравнения используем внешние запросы из `data/synthetic/thyroid_bethesda_examples.csv`; как в TF-IDF, случаи автоматически не исключаются по совпадению текста.


Ниже добавлены прогон 30 запросов и экспорт CSV для ручной проверки с теми же колонками, что у TF-IDF. Ручная релевантность не выводится из Bethesda или similarity. Ниже сохраняем текущий FAISS-индекс и логируем run в локальный MLflow. Метрики качества до общей ручной разметки не рассчитываем; загрузка индекса для повторного поиска отдельным сценарием не реализована.

In [ ]:
def find_similar_cases(text, top_k=5):
    """Вернуть top-k случаев по косинусному сходству описаний."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError('text должен быть непустой строкой.')
    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
        raise ValueError('top_k должен быть положительным целым числом.')
    if index.ntotal != len(search_cases) or index.ntotal == 0:
        raise ValueError('Индекс пуст или не соответствует таблице случаев.')

    query_vector = model.encode(
        [text.strip()], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, index.d) or not np.isfinite(query_vector).all():
        raise ValueError('Некорректный вектор запроса.')
    if not np.allclose(np.linalg.norm(query_vector, axis=1), 1.0, atol=1e-5):
        raise ValueError('Вектор запроса должен иметь единичную норму.')

    scores, positions = index.search(query_vector, min(top_k, index.ntotal))
    results = search_cases.iloc[positions[0]][RESULT_COLUMNS].copy()
    results.insert(0, 'similarity', scores[0])
    results.insert(0, 'rank', np.arange(1, len(results) + 1))
    return results.reset_index(drop=True)


## 8. Пример вызова

Ниже — явно вымышленный запрос для знакомства с интерфейсом, не тест качества. Оставляем его пустым по умолчанию, чтобы последовательный запуск не выводил реальный корпус. Для локального просмотра сначала проверьте допустимость вывода свободного текста. Пример синтаксиса: `find_similar_cases('Вымышленное описание: группы клеток и коллоид.', top_k=5)`.

In [ ]:
query_text = 'Материал представлен содержимым кистозной полости, обильным коллоидом и единичными мелкими скоплениями дистрофичных тиреоцитов.'  # Вставьте отдельно согласованный запрос для локального просмотра.
if query_text.strip():
    results = find_similar_cases(query_text, top_k=5)
    display(results)


## 9. Выбор 30 внешних запросов

Источник — `data/synthetic/thyroid_bethesda_examples.csv`. `calculate_sha256(Path(...))` сохраняет полный хеш файла в `QUERY_SOURCE_SHA256` до чтения. `read_csv` загружает только description и bethesda с теми же параметрами, что `test_TF-IDF.ipynb`. Агент файл не читает и ячейки не выполняет.

Повторяем правила TF-IDF: категории I–VI в заданном порядке; в каждой категории пропускаем отсутствующие и пустые после strip описания, берём первые пять в порядке CSV. Запросы приводятся к strip, как у TF-IDF; исходный корпус остаётся без очистки. Bethesda служит только для выбора запросов, не для оценки выдачи. Если пяти запросов в категории нет, останавливаемся.

In [ ]:
TEST_EXAMPLES_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'synthetic'
    / 'thyroid_bethesda_examples.csv'
)
QUERY_SOURCE_SHA256 = calculate_sha256(Path(TEST_EXAMPLES_PATH))

BETHESDA_CATEGORIES = ['I', 'II', 'III', 'IV', 'V', 'VI']
QUERIES_PER_CATEGORY = 5

test_examples = pd.read_csv(
    TEST_EXAMPLES_PATH, sep=';', encoding='utf-8-sig',
    usecols=['description', 'bethesda'], dtype='string',
)

source_categories = test_examples['bethesda'].str.strip()
test_queries_by_bethesda: dict[str, list[str]] = {}

for category in BETHESDA_CATEGORIES:
    category_mask = source_categories.eq(category).fillna(False)
    descriptions = (
        test_examples.loc[category_mask, 'description']
        .dropna()
        .str.strip()
    )
    descriptions = descriptions.loc[descriptions.ne('')]
    selected_descriptions = descriptions.head(QUERIES_PER_CATEGORY).tolist()

    if len(selected_descriptions) != QUERIES_PER_CATEGORY:
        raise ValueError(
            f'Для Bethesda {category} найдено меньше '
            f'{QUERIES_PER_CATEGORY} непустых описаний.'
        )

    test_queries_by_bethesda[category] = selected_descriptions


## 10. Прогон запросов и таблица результатов

Собираем запросы по категориям I–VI, проверяем ровно 30 и назначаем query_id с 1 через `enumerate(..., start=1)`. Для каждого вызываем существующую `find_similar_cases` с `top_k=20`; модель, векторы, FAISS и ранжирование не меняются. При ошибке запрос не пропускаем молча.

Ожидается до 600 строк: 30 запросов × до 20 результатов. Если индекс содержит меньше 20 случаев, сохраняем только фактическую выдачу без дополнения. `pd.concat` объединяет результаты в `review_table` в порядке запросов и рангов, а `EXPECTED_REVIEW_COLUMNS` задаёт общий с TF-IDF порядок колонок.

Ручные поля пустые. Шкала manual_relevance: 2 — высокая похожесть, 1 — частичная, 0 — нерелевантный результат; пустое — не размечено, а не 0. Bethesda и conclusion — справочные поля, не оценка похожести. Таблица предназначена для будущего объединения с TF-IDF top-20 в общем пуле ручной разметки; объединение и метрики сейчас не выполняются.

In [ ]:
TOP_K = 20
EXPECTED_REVIEW_COLUMNS = [
    'query_id', 'query_description', 'rank', 'similarity',
    'case_id', 'description', 'conclusion', 'bethesda',
    'manual_relevance', 'manual_comment',
]

query_texts = [
    text
    for category in BETHESDA_CATEGORIES
    for text in test_queries_by_bethesda[category]
]
QUERY_COUNT = len(query_texts)
if QUERY_COUNT != 30:
    raise ValueError(f'Ожидалось 30 запросов, выбрано {QUERY_COUNT}.')

review_rows = []
for query_id, query_description in enumerate(query_texts, start=1):
    query_results = find_similar_cases(query_description, top_k=TOP_K).copy()
    query_results.insert(0, 'query_description', query_description)
    query_results.insert(0, 'query_id', query_id)
    query_results['manual_relevance'] = ''
    query_results['manual_comment'] = ''
    review_rows.append(query_results[EXPECTED_REVIEW_COLUMNS])

review_table = pd.concat(review_rows, ignore_index=True)[EXPECTED_REVIEW_COLUMNS]
RETRIEVAL_CONTEXT = {
    'dataset_version_sha256': DATASET_VERSION,
    'query_source_sha256': QUERY_SOURCE_SHA256,
    'index_sha256': INDEX_METADATA['index_sha256'],
    'query_count': QUERY_COUNT,
    'top_k': TOP_K,
}
print({'query_count': QUERY_COUNT, 'returned_rows': len(review_table)})


## 11. Проверка версий и экспорт CSV

Перед экспортом повторно вычисляем SHA256 корпуса и файла запросов, сравниваем с DATASET_VERSION и QUERY_SOURCE_SHA256. При изменении любого файла экспорт останавливается: перезапустите ноутбук сверху вниз для согласованной выдачи.

`output_path.parent.mkdir(parents=True, exist_ok=True)` создаёт нужную папку, если её нет. `review_table.to_csv` сохраняет `data/processed/search_results/sentence_transformers_30_queries_results.csv` с `sep=';'`, `encoding='utf-8-sig'`, `index=False`. Колонки проверяем перед записью. Таблицы и тексты не выводим.

Повторный запуск перезаписывает только сгенерированный CSV. Ручную разметку ведите в отдельной копии `sentence_transformers_30_queries_results_labeled.csv`: этот код её не создаёт и не перезаписывает. Не размечайте автоматически перезаписываемый файл. В следующих шагах сохраняем индекс и JSON отдельно от CSV; размеченные копии в MLflow не отправляются.

In [ ]:
output_path = (
    PROJECT_ROOT / 'data' / 'processed' / 'search_results'
    / 'sentence_transformers_30_queries_results.csv'
)
if list(review_table.columns) != EXPECTED_REVIEW_COLUMNS:
    raise ValueError('Колонки выдачи не соответствуют ожидаемому формату CSV.')
if calculate_sha256(Path(DATA_PATH)) != DATASET_VERSION:
    raise ValueError('Корпус изменился после загрузки; перезапустите ноутбук сверху вниз.')
if calculate_sha256(Path(TEST_EXAMPLES_PATH)) != QUERY_SOURCE_SHA256:
    raise ValueError('Источник запросов изменился после загрузки; перезапустите ноутбук сверху вниз.')

output_path.parent.mkdir(parents=True, exist_ok=True)
review_table.to_csv(output_path, sep=';', encoding='utf-8-sig', index=False)
RESULTS_FILE_SHA256 = calculate_sha256(Path(output_path))
RESULTS_ROW_COUNT = len(review_table)
print({'exported_rows': len(review_table)})


## 12. Проверки перед сохранением и логированием

Сначала сохраните код и сделайте commit самостоятельно, затем перезапустите ядро и выполните ноутбук сверху вниз. Полный Git commit и dirty-состояние получаем относительно PROJECT_ROOT. Имя участника проверьте в настройках MLflow ниже.

validate_run_inputs() проверяет неизменность корпуса и запросов, 30 запросов, top-20, структуру фактически сохранённого CSV и его хеш, число строк и пустые ручные поля. Для текущего индекса проверяются CPU-тип IndexFlatIP, размерность, число векторов, порядок case_id и отпечаток векторов. Настройки модели и устройства сравниваются со снимком при её загрузке и построении индекса. Ошибки останавливают сохранение и логирование без исправления метаданных.

SHA256 фиксирует версию байтов файла. Устройство и seed записываются в run, но seed не обеспечивает побитовое совпадение CPU/CUDA. Эти проверки не измеряют клиническое качество.


In [ ]:
def validate_run_inputs():
    """Проверить согласованность входов, текущего индекса и созданной выдачи."""
    if calculate_sha256(Path(DATA_PATH)) != DATASET_VERSION:
        raise ValueError('Корпус изменился; перезапустите ноутбук сверху вниз.')
    if calculate_sha256(Path(TEST_EXAMPLES_PATH)) != QUERY_SOURCE_SHA256:
        raise ValueError('Файл запросов изменился; перезапустите ноутбук сверху вниз.')
    selected_query_count = sum(len(texts) for texts in test_queries_by_bethesda.values())
    if QUERY_COUNT != 30 or len(query_texts) != 30 or selected_query_count != 30 or TOP_K != 20:
        raise ValueError('Для этого run нужны ровно 30 запросов и top_k=20.')
    expected_context = {
        'dataset_version_sha256': DATASET_VERSION,
        'query_source_sha256': QUERY_SOURCE_SHA256,
        'index_sha256': INDEX_METADATA['index_sha256'],
        'query_count': QUERY_COUNT,
        'top_k': TOP_K,
    }
    if RETRIEVAL_CONTEXT != expected_context:
        raise ValueError('Параметры изменились после поиска; перезапустите ноутбук сверху вниз.')
    if list(review_table.columns) != EXPECTED_REVIEW_COLUMNS or not output_path.is_file():
        raise ValueError('Отсутствует CSV выдачи или неверны колонки review_table.')
    csv_columns = pd.read_csv(output_path, sep=';', encoding='utf-8-sig', nrows=0).columns.tolist()
    if csv_columns != EXPECTED_REVIEW_COLUMNS:
        raise ValueError('Колонки фактического CSV не соответствуют ожидаемым.')
    if calculate_sha256(Path(output_path)) != RESULTS_FILE_SHA256:
        raise ValueError('CSV изменился после экспорта; изменённый файл не логируем.')
    if len(review_table) != RESULTS_ROW_COUNT or RESULTS_ROW_COUNT != 30 * min(20, index.ntotal):
        raise ValueError('Размер выдачи не соответствует выполненному прогону.')
    if not review_table[['manual_relevance', 'manual_comment']].eq('').all().all():
        raise ValueError('Этот run ожидает пустые ручные поля и awaiting_manual_labels.')
    case_ids = search_cases['case_id'].tolist()
    if index.ntotal != len(case_ids) or case_ids != INDEX_METADATA['case_ids']:
        raise ValueError('Размер индекса или порядок case_id изменился.')
    if type(index) is not faiss.IndexFlatIP or index.metric_type != faiss.METRIC_INNER_PRODUCT:
        raise ValueError('Ожидается CPU faiss.IndexFlatIP со скалярным произведением.')
    actual_settings = {
        'model_name': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'device': str(model.device),
        'batch_size': BATCH_SIZE,
        'max_seq_length': model.max_seq_length,
        'embedding_dimension': model.get_embedding_dimension(),
        'random_seed': SEED,
    }
    if model.device.type == 'cuda':
        actual_settings['gpu_name'] = torch.cuda.get_device_name(model.device)
    if actual_settings != MODEL_SETTINGS or model.device.type != torch.device(DEVICE).type:
        raise ValueError('Настройки модели или устройство изменились после загрузки.')
    expected_metadata = {
        **actual_settings,
        'dataset_version_sha256': DATASET_VERSION,
        'embedding_dimension': index.d,
        'index_size_cases': index.ntotal,
        'normalize_embeddings': True,
        'faiss_index_type': type(index).__name__,
        'faiss_device': 'cpu',
        'case_ids': case_ids,
        'index_sha256': hashlib.sha256(faiss.serialize_index(index)).hexdigest(),
    }
    if index.d != model.get_embedding_dimension() or INDEX_METADATA != expected_metadata:
        raise ValueError('Индекс не соответствует зафиксированным метаданным; не перезаписываем их.')


validate_run_inputs()


## 13. Экспорт использованного FAISS-индекса

Сохраняем текущий index через faiss.write_index и JSON в data/processed/search_indexes/. Имена включают полный SHA256 корпуса, revision модели и фактическое устройство encoder. JSON содержит настройки, число и размерность векторов, нормировку, тип и CPU-устройство FAISS, хеш индекса и упорядоченные case_id; описаний и заключений нет.

Если пара файлов уже существует, используем её только при полном совпадении JSON и хеша бинарного файла с текущим индексом. Несоответствие или неполная пара останавливает работу: старые метаданные не перезаписываются. Новый индекс не строится для логирования. Отдельную матрицу эмбеддингов и веса модели не сохраняем.

Сценарий загрузки готового индекса для повторного поиска не добавлен: существующий файл здесь только проверяется и используется как артефакт.


In [ ]:
validate_run_inputs()
INDEX_DIRECTORY = PROJECT_ROOT / 'data' / 'processed' / 'search_indexes'
INDEX_BASENAME = (
    f"sentence_transformers_{DATASET_VERSION}_{MODEL_REVISION}_"
    f"{str(model.device).replace(':', '_')}"
)
INDEX_BINARY_PATH = INDEX_DIRECTORY / f'{INDEX_BASENAME}.faiss'
INDEX_METADATA_PATH = INDEX_DIRECTORY / f'{INDEX_BASENAME}.json'

if INDEX_BINARY_PATH.exists() != INDEX_METADATA_PATH.exists():
    raise ValueError('Пара файлов индекса неполная; существующие файлы не перезаписываем.')
if not INDEX_BINARY_PATH.exists():
    INDEX_DIRECTORY.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(INDEX_BINARY_PATH))
    with INDEX_METADATA_PATH.open('x', encoding='utf-8') as metadata_file:
        json.dump(INDEX_METADATA, metadata_file, ensure_ascii=False, indent=2)
        metadata_file.write('\n')


def validate_index_artifacts():
    """Проверить сохранённую пару без перезаписи или восстановления индекса."""
    if not INDEX_BINARY_PATH.is_file() or not INDEX_METADATA_PATH.is_file():
        raise FileNotFoundError('Отсутствуют файлы FAISS-индекса или JSON метаданных.')
    with INDEX_METADATA_PATH.open(encoding='utf-8') as metadata_file:
        saved_metadata = json.load(metadata_file)
    if saved_metadata != INDEX_METADATA:
        raise ValueError('JSON старого индекса не соответствует текущему; перезапись запрещена.')
    if calculate_sha256(Path(INDEX_BINARY_PATH)) != INDEX_METADATA['index_sha256']:
        raise ValueError('Файл индекса не соответствует текущему индексу или был изменён.')


validate_index_artifacts()
print({'saved_index_vectors': index.ntotal})


## 14. Параметры MLflow

Общие ключи совпадают с test_TF-IDF: происхождение данных, запросов, кода и CSV; параметры поиска; размеры индекса и запросов; статус awaiting_manual_labels. Добавляем закреплённую модель, устройство encoder и GPU при CUDA, seed, параметры кодирования и версии библиотек. FAISS фиксируется как CPU IndexFlatIP.

calculate_sha256 вычисляет все файловые хеши. subprocess.run получает полный Git SHA и dirty-состояние относительно PROJECT_ROOT; ошибки Git не скрываются. Проверки повторяем перед логированием. Параметры записываются в params без дублирования в tags. [API логирования MLflow](https://mlflow.org/docs/latest/python_api/mlflow.html).


In [ ]:
from datetime import datetime, timezone
import platform
import subprocess

import mlflow

MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
MLFLOW_EXPERIMENT_NAME = 'semantic-case-search'
PARTICIPANT_NAME = 'Vildan'

if not PARTICIPANT_NAME.strip() or PARTICIPANT_NAME.startswith('<SET_'):
    raise ValueError('Укажите имя участника перед логированием.')
validate_run_inputs()
validate_index_artifacts()

GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT,
    check=True, capture_output=True, text=True, encoding='utf-8',
).stdout.strip()
if not re.fullmatch(r'[0-9a-fA-F]{40}', GIT_COMMIT):
    raise ValueError('Git не вернул полный 40-символьный commit SHA.')
GIT_WORKTREE_DIRTY = bool(subprocess.run(
    ['git', 'status', '--porcelain=v1', '--untracked-files=normal'],
    cwd=PROJECT_ROOT, check=True, capture_output=True, text=True, encoding='utf-8',
).stdout.strip())

RUN_DATE_UTC = datetime.now(timezone.utc).isoformat(timespec='seconds')
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'search' / 'test_sentence-transformers.ipynb'
REVIEW_CSV_PATH = output_path
INDEX_SIZE_CASES = int(index.ntotal)
RUN_PARAMETERS = {
    'participant_name': PARTICIPANT_NAME,
    'task_name': 'similar_case_retrieval',
    'run_date_utc': RUN_DATE_UTC,
    'git_commit': GIT_COMMIT,
    'git_worktree_dirty': GIT_WORKTREE_DIRTY,
    'method': 'Sentence Transformers cosine retrieval',
    'dataset_file': DATA_PATH.name,
    'dataset_version_sha256': DATASET_VERSION,
    'query_source_file': TEST_EXAMPLES_PATH.name,
    'query_source_sha256': QUERY_SOURCE_SHA256,
    'results_file': REVIEW_CSV_PATH.name,
    'retrieval_examples_sha256': calculate_sha256(Path(REVIEW_CSV_PATH)),
    'notebook_sha256': calculate_sha256(Path(NOTEBOOK_PATH)),
    'index_size_cases': INDEX_SIZE_CASES,
    'query_count': QUERY_COUNT,
    'top_k': TOP_K,
    'query_field': 'description',
    'similarity_metric': 'cosine',
    'self_match_exclusion': 'none; external queries; no automatic text matching',
    'evaluation_status': 'awaiting_manual_labels',
    **MODEL_SETTINGS,
    'normalize_embeddings': INDEX_METADATA['normalize_embeddings'],
    'faiss_index_type': INDEX_METADATA['faiss_index_type'],
    'faiss_device': 'cpu',
    'index_file_sha256': calculate_sha256(Path(INDEX_BINARY_PATH)),
    'index_metadata_sha256': calculate_sha256(Path(INDEX_METADATA_PATH)),
    'python_version': platform.python_version(),
    'torch_version': str(torch.__version__),
}
RUN_PARAMETERS.update({
    f'{distribution.replace("-", "_")}_version': version(distribution)
    for distribution in (
        'sentence-transformers', 'transformers', 'huggingface-hub',
        'faiss-cpu', 'numpy', 'pandas', 'mlflow',
    )
})


## 15. Запись run и артефактов

Выбираем эксперимент по имени semantic-case-search на http://127.0.0.1:5000. Run: sentence-transformers-top20-30-queries-results. Отправляем только фактически использованный FAISS-файл, его проверенный JSON и созданный CSV 30 запросов. Исходный prepared_dataset.csv, отдельная матрица эмбеддингов и веса модели в MLflow не отправляются.

returned_cases — технический счётчик строк. Метрики качества до общей ручной разметки не логируем. Сравнение методов будет выполнено позднее по общему пулу; фактическое устройство encoder остаётся в params. После успешной записи выводим run_id. Подготовка кода не подтверждает, что сервер доступен или что запуск зарегистрирован.


In [ ]:
validate_run_inputs()
validate_index_artifacts()
if calculate_sha256(Path(NOTEBOOK_PATH)) != RUN_PARAMETERS['notebook_sha256']:
    raise ValueError('Код ноутбука изменился после подготовки параметров.')
if calculate_sha256(Path(INDEX_METADATA_PATH)) != RUN_PARAMETERS['index_metadata_sha256']:
    raise ValueError('JSON индекса изменился после подготовки параметров.')
if mlflow.active_run() is not None:
    raise RuntimeError('Уже открыт другой MLflow run; завершите его перед этим запуском.')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
with mlflow.start_run(run_name='sentence-transformers-top20-30-queries-results') as active_run:
    mlflow.log_params(RUN_PARAMETERS)
    mlflow.log_metric('returned_cases', len(review_table))
    mlflow.log_artifact(str(INDEX_BINARY_PATH), artifact_path='model')
    mlflow.log_artifact(str(INDEX_METADATA_PATH), artifact_path='model')
    mlflow.log_artifact(str(REVIEW_CSV_PATH), artifact_path='retrieved_examples')

print(f'Запуск MLflow сохранён. Run ID: {active_run.info.run_id}')


## Что эта реализация позволяет проверить

После разрешённой подготовки запуска: сначала на отдельно согласованном synthetic-корпусе проверить соответствие позиций строкам, порядок оценок, `top_k` больше размера корпуса и отклонение пустого запроса. Численные проверки в ячейках контролируют структуру индекса; они не доказывают клиническую релевантность.

Следующий эксперимент требует отдельного согласования. Гипотеза: семантические векторы лучше TF-IDF находят похожие описания при разных формулировках. Baseline — существующий TF-IDF; основной изменяемый фактор — представление текста. Для проверки нужны одни и те же независимые запросы, определение клинической похожести и ручная оценка. Совпадение Bethesda не заменяет релевантность. Для Recall@5/10 нужен эталон релевантных случаев, а не только оценка top-k выдачи. До получения эталона нельзя заявить улучшение.

Учёт подготовленной выдачи предусмотрен в локальном MLflow со статусом awaiting_manual_labels. Фактический запуск агентом не выполнялся. Сравнение качества остаётся следующим отдельным этапом по общей ручной разметке; критерий принятия модели ещё требует согласования.

**Проверка понимания:** объясните своими словами путь от описания до найденных случаев. Почему для `IndexFlatIP` мы нормируем и корпус, и запрос? Почему после построения индекса нельзя переставлять строки `search_cases`?